# Семинар 9 - Методы построения оптического потока по последовательности изображений

**Этот семинар содержит оцениваемое домашнее задание**

***

Источник - https://habr.com/ru/post/201406/

$\textbf{Task statement}$: Оптический поток (ОП) – изображение видимого движения, представляющее собой сдвиг каждой точки (пикселя) между двумя изображениями.

По сути, он представляет собой поле скоростей. Суть ОП в том, что для каждой точки изображения $I_{t_0} (\vec{r})$ находится такой вектор сдвига $\delta \vec{r}$, чтобы было соответсвие между исходной точкой и точкой на следущем фрейме $I_{t_1} (\vec{r} + \delta \vec{r})$. В качестве метрики соответвия берут близость интенсивности пикселей, беря во внимание маленькую разницу по времени между кадрами: $\delta{t} = t_{1} - t_{0}$. В более точных методах точку можно привязывать к объекту на основе, например, выделения ключевых точек, а также считать градиенты вокруг точки, лапласианы и проч.

$\textbf{For what}$: Определение собственной скорости, Определение локализации, Улучшение методов трекинга объектов, сегментации, Детектирование событий, Сжатие видеопотока и проч.

![](data/tennis.png)

Разделяют 2 вида оптического потока - плотный (dense) [Farneback method, neural nets], работающий с целым изображением, и выборочный (sparse) [Lucas-Kanade method], работающий с ключевыми точками

In [ ]:
import cv2
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import IPython

%matplotlib inline

## Lucas-Kanade (sparse)

Пусть $I_{1} = I(x, y, t_{1})$ интенсивность в некоторой точке (x, y) на первом изображении (т. е. в момент времени t). На втором изображении эта точка сдвинулась на (dx, dy), при этом прошло время dt, тогда $I_{2} = I(x + dx, y + dx, t_{1} + dt) \approx I_{1} + I_{x}dx + I_{y}dy +  I_{t}dt$. Из постановки задачи следует, что интенсивность пикселя не изменилась, тогда $I_{1} = I_{2}$. Далее определяем $dx, dy$.

Самое простое решение проблемы – алгоритм Лукаса-Канаде. У нас же на изображении объекты размером больше 1 пикселя, значит, скорее всего, в окрестности текущей точки у других точек будут примерно такие же сдвиги. Поэтому мы возьмем окно вокруг этой точки и минимизируем (по МНК) в нем суммарную погрешность с весовыми коэффициентами, распределенными по Гауссу, то есть так, чтобы наибольший вес имели пиксели, ближе всего находящиеся к исследуемому.

**Полезные материалы:** 
- цикл видео-лекций от First Principles of Computer Vision, посвященный Optical Flow и алгоритму Lucas-Kanade: https://youtube.com/playlist?list=PL2zRqk16wsdoYzrWStffqBAoUY8XdvatV

### Вопрос 1

Перечислите три основных предположения, на которых базируется метод Lucas-Kanade. Почему каждое из них важно для корректной работы алгоритма?

**Ответ:**
1. Непрерывность (постоянность) яркости - интенсивности пикселей остаются неизменными между последовательными кадрами. Необходимо, чтобы установить соответствия между пикселями.
2. Малое перемещение - смещение между кадрами должно быть достаточно малым для того, чтобы линейное приближение первого порядка ряда Тейлора было корректным.
3. Пространственная согласованность (гладкость смещений) - соседние пиксели движутся схожим образом (имеют близкие скорости). Позволяет использовать небольшое окно пикселей для решения недоопределенной системы уравнений.

### Вопрос 2

Объясните, зачем нужен пирамидальный подход в алгоритме Lucas-Kanade. Какую проблему он решает и как именно?

**Ответ:** Он решает проблему ограничения на малое перемещение, которое может быть недостаточным для отслеживания быстрых движений объектов. Метод работает путем построения пирамиды изображений с уменьшающимся разрешением, где сначала вычисляется приблизительное движение на верхнем уровне пирамиды с меньшим разрешением. Затем это приближение уточняется на каждом следующем уровне, что позволяет отслеживать большие перемещения, сохраняя точность алгоритма.

### Вопрос 3

С какими проблемами может столкнуться алгоритм Lucas-Kanade при отслеживании точек на видео? Назовите минимум три ограничения.

**Ответ:**
1. Нарушение работы из-за быстрого перемещения объектов (нарушение предположения о малом перемещении)
2. Резкое изменение яркости из-за бликов, мерцаний, появления теней (нарушение предположения о постоянстве яркости).
3. Отслеживание "шумных" объектов – снег, вода, артефакты видеозаписи.
4. Перекрытие объектов - если один объект закрывает отслеживаемую точку другого, то он может ее "перехватить".


### Задание 1

Напишите реализацию Лукаса-Канаде c помощью numpy и cv2. Сравните с реализацией `cv2.calcOpticalFlowPyrLK`.

In [ ]:
def build_image_pyramid(image, num_levels, scale_factor=0.5):
    pyramid = [image.copy()]
    h, w = image.shape[:2]
    for level in range(1, num_levels):
        factor = scale_factor ** level
        new_size = (int(w * factor), int(h * factor))
        pyramid.append(cv2.resize(pyramid[0], new_size, interpolation=cv2.INTER_LINEAR))
    return pyramid


def compute_image_gradients(image):
    Ix = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    return Ix, Iy


def compute_lk_optical_flow_point(Ix, Iy, It, window_size=5):
    ix2 = Ix * Ix
    iy2 = Iy * Iy
    ixiy = Ix * Iy
    sum_ix2 = ix2.sum()
    sum_iy2 = iy2.sum()
    sum_ixiy = ixiy.sum()
    a = np.array([[sum_ix2, sum_ixiy], [sum_ixiy, sum_iy2]])
    b = -np.array([Ix.dot(It).sum(), Iy.dot(It).sum()])
    try:
        if np.linalg.cond(a) > 1e4:
            return None, None
        ux, uy = np.linalg.solve(a, b)
        return ux, uy
    except np.linalg.LinAlgError:
        return None, None


def compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size=5):
    Ix, Iy = compute_image_gradients(prev_patch)
    It = (curr_patch - prev_patch).astype(np.float64)
    return compute_lk_optical_flow_point(Ix, Iy, It, window_size)


def track_point_with_pyramid_lk(prev_pyramid, curr_pyramid, point, window_size=15, max_iterations=10, epsilon=0.01):
    num_levels = len(prev_pyramid)
    current_point = point.copy()

    for level in range(num_levels-1, -1, -1):
        scale = 0.5 ** level
        scaled_point = current_point * scale

        half_window = window_size // 2
        h, w = prev_pyramid[level].shape

        if (scaled_point[0] < half_window or scaled_point[0] >= w - half_window or
            scaled_point[1] < half_window or scaled_point[1] >= h - half_window):
            return None

        x, y = int(scaled_point[0]), int(scaled_point[1])
        prev_patch = prev_pyramid[level][y-half_window:y+half_window+1,
                                       x-half_window:x+half_window+1]
        curr_patch = curr_pyramid[level][y-half_window:y+half_window+1,
                                       x-half_window:x+half_window+1]

        for _ in range(max_iterations):
            u, v = compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size)

            if u is None or v is None:
                return None

            if abs(u) < epsilon and abs(v) < epsilon:
                break

            scaled_point[0] += u
            scaled_point[1] += v

            x, y = int(scaled_point[0]), int(scaled_point[1])
            if (x < half_window or x >= w - half_window or
                y < half_window or y >= h - half_window):
                return None

            prev_patch = prev_pyramid[level][y-half_window:y+half_window+1,
                                           x-half_window:x+half_window+1]
            curr_patch = curr_pyramid[level][y-half_window:y+half_window+1,
                                           x-half_window:x+half_window+1]

        if level > 0:
            current_point = scaled_point / scale

    return current_point


def lucas_kanade_optical_flow(prev_frame, curr_frame, points,
                             window_size=15, num_pyramid_levels=3,
                             max_iterations=10, epsilon=0.01):
    if len(prev_frame.shape) == 3:
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
    else:
        prev_gray = prev_frame
        curr_gray = curr_frame

    prev_gray = prev_gray.astype(np.float32)
    curr_gray = curr_gray.astype(np.float32)

    prev_pyramid = build_image_pyramid(prev_gray, num_pyramid_levels)
    curr_pyramid = build_image_pyramid(curr_gray, num_pyramid_levels)

    new_points = np.zeros_like(points)
    status = np.zeros(len(points), dtype=np.uint8)

    for i, point in enumerate(points):
        if (point[0] < 0 or point[0] >= prev_gray.shape[1] or
            point[1] < 0 or point[1] >= prev_gray.shape[0]):
            new_points[i] = point
            status[i] = 0
            continue

        tracked_point = track_point_with_pyramid_lk(
            prev_pyramid, curr_pyramid, point,
            window_size, max_iterations, epsilon
        )

        if tracked_point is not None:
            if 0 <= tracked_point[0] < curr_gray.shape[1] and 0 <= tracked_point[1] < curr_gray.shape[0]:
                new_points[i] = tracked_point
                status[i] = 1
            else:
                new_points[i] = point
                status[i] = 0
        else:
            new_points[i] = point
            status[i] = 0

    return new_points, status


def demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    cap = cv2.VideoCapture(video_path)

    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    p0 = p0.reshape(-1, 2)

    initial_points = p0.copy()

    mask = np.zeros_like(old_frame)

    color = np.random.randint(0, 255, (len(p0), 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        p1, st = lucas_kanade_optical_flow(
            old_gray,
            frame_gray,
            p0,
            window_size=15,
            num_pyramid_levels=3
        )

        good_new = p1[st == 1]
        good_old = p0[st == 1]

        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new
            c, d = old
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        img = cv2.add(frame, mask)

        out.write(img)

        old_gray = frame_gray.copy()

        p0[st == 1] = good_new

    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [ ]:
result_path = demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4')

### Релизация OpenCV - cv2.calcOpticalFlowPyrLK

In [ ]:
def demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    cap = cv2.VideoCapture(video_path)

    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    lk_params = dict(
        winSize=(15, 15),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
    )

    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

    mask = np.zeros_like(old_frame)

    color = np.random.randint(0, 255, (100, 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new.ravel()
            c, d = old.ravel()
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        img = cv2.add(frame, mask)

        out.write(img)

        old_gray = frame_gray.copy()

        p0 = good_new.reshape(-1, 1, 2)

    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [ ]:
result_path = demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_LK.mp4')

### Задание 2

В базовой реализации у кода есть одна важная проблема - ключевые точки инициализируются единожды. В реальных задачах необходимо отслеживать точки, которые исчезают из кадра и появляются в других местах. Реализуйте механизм, который будет отслеживать точки, которые пропадают из кадра и добавлять новые точки в те места, где они появляются. Для этого вам нужно будет реализовать механизм поиска новых точек на изображении.

In [ ]:
def find_new_points(image, existing_points, max_points=100, min_distance=20, quality_level=0.01):
    mask = np.ones(image.shape[:2], dtype=np.uint8) * 255

    for point in existing_points:
        x, y = int(point[0]), int(point[1])
        cv2.circle(mask, (x, y), min_distance, 0, -1)

    new_points = cv2.goodFeaturesToTrack(
        image,
        maxCorners=max_points,
        qualityLevel=quality_level,
        minDistance=min_distance,
        mask=mask
    )

    if new_points is not None:
        return new_points.reshape(-1, 2)
    return np.array([])

def demo_optical_flow_with_point_update(video_path='data/slow_traffic_small.mp4',
                                      output_path='output_my_LK_with_update.mp4'):
    cap = cv2.VideoCapture(video_path)

    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    p0 = p0.reshape(-1, 2)

    mask = np.zeros_like(old_frame)

    color = np.random.randint(0, 255, (len(p0), 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):
        ret, frame = cap.read()
        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        p1, st = lucas_kanade_optical_flow(
            old_gray,
            frame_gray,
            p0,
            window_size=15,
            num_pyramid_levels=3,
        )

        good_new = p1[st == 1]
        good_old = p0[st == 1]

        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new
            c, d = old
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)),
                          color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5,
                             color[i % len(color)].tolist(), -1)

        p0 = good_new.copy()

        if len(p0) < feature_params['maxCorners'] // 2:
            new_points = find_new_points(
                frame_gray,
                p0,
                max_points=feature_params['maxCorners'] - len(p0),
                min_distance=feature_params['minDistance'],
                quality_level=feature_params['qualityLevel']
            )
            if len(new_points) > 0:
                p0 = np.vstack((p0, new_points))
                new_colors = np.random.randint(0, 255, (len(new_points), 3))
                color = np.vstack((color, new_colors))

        img = cv2.add(frame, mask)

        out.write(img)

        old_gray = frame_gray.copy()

    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [ ]:
result_path = demo_optical_flow_with_point_update(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK_with_update.mp4')

### Вопрос 4

В чем основное отличие разреженного (sparse) оптического потока Lucas-Kanade от плотного (dense) оптического потока (например, метода Farneback)?

**Ответ:** Разреженный оптический поток Lucas-Kanade вычисляет перемещения только для специально выбранных характерных точек (например, углов), что делает его более эффективным и устойчивым к шумам, но не дает полной информации о движении всех пикселей изображения. В отличие от этого, плотный оптический поток (метод Farneback) вычисляет векторы движения для каждого пикселя изображения, что дает более полную информацию о движении, но требует больше вычислительных ресурсов и может быть менее устойчивым к шумам и изменениям освещения.


## Farneback (dense)

Метод Farneback носит несколько более глобальный характер, чем метод Лукаса-Канаде. Он опирается на предположение о том, что на всем изображении оптический поток будет достаточно гладким.

# Вопрос 5

Перечислите основные шаги алгоритма Farneback для расчета оптического потока.

**Ответ:**
1. Построение пирамиды изображений для обработки больших перемещений.
2. Аппроксимация локальных областей полиномами второго порядка.
3. Вычисление коэффициентов полиномов для обоих кадров.
4. Решение системы уравнений для получения векторов перемещения с учетом предположения о гладкости потока.
5. Итеративное уточнение результатов на каждом уровне пирамиды от верхнего к нижнему.

### Вопрос 6

Каким образом в методе Farneback обрабатываются большие смещения объектов между кадрами?

**Ответ:** С помощью пирамидального подхода. Сначала вычисляется приблизительное движение на верхнем уровне пирамиды с меньшим разрешением, а затем это приближение последовательно уточняется на каждом следующем уровне. При переходе на более высокий уровень пирамиды большие смещения становятся относительно меньше, что позволяет алгоритму корректно их обрабатывать, а затем полученное приближение масштабируется и уточняется при спуске по уровням пирамиды.


In [ ]:
def demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_Farneback.mp4'):
    cap = cv2.VideoCapture(video_path)

    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    ret, frame1 = cap.read()
    if not ret:
        print('Не удалось прочитать видео')
        return None

    prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)

    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):
        ret, frame2 = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        next_frame = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            prvs, next_frame, None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])

        hsv[..., 0] = ang * 180 / np.pi / 2

        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)

        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        out.write(bgr)

        prvs = next_frame

    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [ ]:
result_path = demo_optical_flow_farneback_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_farneback.mp4')

### Вопрос 7

Как влияет предварительная обработка изображений (фильтрация шума, выравнивание гистограмм) на качество оптического потока, получаемого методом Farneback? Предложите оптимальный пайплайн предобработки.

**Ответ:** Предварительная обработка изображений значительно улучшает качество оптического потока методом Farneback: фильтрация шума (например, гауссовым фильтром) уменьшает влияние случайных помех, а выравнивание гистограмм улучшает контраст и делает градиенты более выраженными.

Предложенный пайплайн:
1. Нормализация яркости
2. Применение Gaussian Blur для уменьшения шума,
3. Выравнивание гистограммы для улучшения контраста.
4. Нормализация градиентов для стабилизации вычислений.